In [8]:
import pandas as pd
import os
import run as run
import chephy_model as cpm
from tqdm import tqdm
from pathlib import Path


In [9]:
def main(file_name, input_file):


    out_folder = f"../output_orr/{file_name}"
    out_name = "neighbors"
    params_file = "stats.csv"
    right = 4
    left = 4
    # label_header = args.epitope_header
    cdr3_header = "aminoAcid"


    # distances_csv = "../distance_matrix.csv"
    # distances_df = pd.read_csv(distances_csv, index_col=0)
    # substitution_matrix = {(aa1, aa2): distances_df.loc[aa1, aa2] for aa1 in distances_df.index for aa2 in distances_df.columns}
    
 
    data = run.load_dataframe(input_file)

    # data = read_csv_files_from_folder()

    data = run.prepare_data(data, cdr3_header, "label_header")    
    cdr3 = list(data[cdr3_header])
    
    sequences_set, full_to_trunc_map = cpm.truncate_sequences(cdr3, right, left)
    chephy_matrix, hamming_matrix, sequences_list = cpm.find_che_phy_dist(sequences_set)


    path = Path(f"{out_folder}")
    path.mkdir(parents=True, exist_ok=True)

    mst = run.build_mst(chephy_matrix, hamming_matrix, sequences_list, full_to_trunc_map, f"{out_folder}/mst.csv")

    # df_results = cluster(mst, out_folder, data, label_header)
    # save_best_cluster(df_results, out_folder)
    # # think of a general method to write files

In [ ]:

folder_path="/dsi/scratch/home/dsi/solefroni/orforchen/downsamples_clonotype_21355"
csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
dataframes = []
num_files = len(csv_files)
i = 1
for file in tqdm(csv_files):
    # print(file, f"file {i} of {num_files}")
    full_file = os.path.join(folder_path, file)
    if os.path.exists(f"../output_orr/{file}"):
        continue                   
    main(file, full_file)
    i+=1

In [ ]:
def traverse_tree(tree, sequence, max_diff, substitution_matrix, path="", current_diff=0, chephy_dist=0, depth=0, GREATEST_DISTANCE=1):
    GREATEST_DISTANCE = 72.38373253708322
    if current_diff > max_diff:
        return []

    if not tree:
        # Normalize chephy_dist by GREATEST_DISTANCE
        normalized_chephy_dist = chephy_dist / GREATEST_DISTANCE if GREATEST_DISTANCE else chephy_dist
        return [(path, current_diff, normalized_chephy_dist)] if current_diff <= max_diff else []

    matching_sequences = []
    for char, subtree in tree.items():
        if depth < len(sequence):
            seq_char = sequence[depth]
            substitution_cost = substitution_matrix.get((seq_char, char), float('inf'))  # Get cost from matrix or set to high cost if undefined
            new_diff = current_diff + (1 if char != seq_char else 0)
            new_chephy_dist = chephy_dist + (substitution_cost if char != seq_char else 0)
        else:
            new_diff = current_diff + 1  # Extra penalty for inserting a character
            new_chephy_dist = chephy_dist + substitution_matrix.get((None, char), 1)  # Cost of insertion (default 1)

        matching_sequences.extend(
            traverse_tree(subtree, sequence, max_diff, substitution_matrix, path + char, new_diff, new_chephy_dist, depth + 1, GREATEST_DISTANCE)
        )

    return matching_sequences

In [ ]:
import pandas as pd

def compute_all_distances_matrix(list_of_sequences, tree, max_diff, substitution_matrix):
    GREATEST_DISTANCE = 72.38373253708322  # Normalization factor

    matrix = {}

    for seq in tqdm(list_of_sequences):
        matrix[seq] = {}
        # Get all possible distances from seq1 to seq2
        results = traverse_tree(tree, seq, max_diff, substitution_matrix)
        for result in results:
            matrix[seq][result[0]] = [result[1], result[2]]

    # Convert to Pandas DataFrame for easier visualization
    df = pd.DataFrame(matrix)
    
    return df


In [ ]:
import chephy_model_tree_with_matrix as cpm
import pandas as pd
from tqdm import tqdm 


distances_csv = "../distance_matrix.csv"
distances_df = pd.read_csv(distances_csv, index_col=0)
substitution_matrix = {(aa1, aa2): distances_df.loc[aa1, aa2] for aa1 in distances_df.index for aa2 in distances_df.columns}
    

file_name = "/dsi/scratch/home/dsi/solefroni/orforchen/downsamples_clonotype_21355/STA_085.csv"
df = pd.read_csv(file_name)
sequences = df["aminoAcid"]
sequences_set, full_to_trunc_map = cpm.truncate_sequences(sequences)
sequences_list = list(sequences_set)
tree = cpm.create_tree(sequences_list)
# compute_distance_matrices(list(sequences_set), substitution_matrix, max_diff=8)
sequences_dict = {"ASNRKGPQ":[]}
compute_all_distances_matrix(sequences_list,tree, 8, substitution_matrix)

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
import pandas as pd

In [ ]:
def prepare_data(data, cdr3_header):

    # understand tcrscapes data preparation


    # remove cdr3 sequences that contain non-aa letters (this also removes nan)
    amino_acid_pattern = r'^[ARNDCEQGHILKMFPSTWYV]+$'

    data_filtered = data[data[cdr3_header].str.match(amino_acid_pattern, case=False, na=False)]

    data_filtered = data_filtered[data_filtered[cdr3_header].str.len() >= 8]

    # drop duplicates
    data_filtered = data_filtered.drop_duplicates(subset=cdr3_header)

    # move truncate to here

    return data_filtered

In [ ]:
sub_matrix = pd.read_csv("../distance_matrix.csv", index_col=0)
sub_matrix = sub_matrix.to_dict()

In [ ]:
import chephy_model as cpm
file_name = "../files/vdjdb_score3.csv"
file_name = "/dsi/scratch/home/dsi/solefroni/orforchen/downsamples_clonotype_21355/STA_085.csv"
df = pd.read_csv(file_name)
# sequences = df["cdr3"]
# sequences = df["aminoAcid"]

cdr3_header = "aminoAcid"

# data = prepare_data(df, cdr3_header)    
# cdr3 = list(data[cdr3_header])
cdr3 = ["CASSFQGYTEAFF","CASSFQGFTEAFF"]
sequences_set, full_to_trunc_map = cpm.truncate_sequences(cdr3)
sequences = list(sequences_set)

In [ ]:
sequences

In [10]:
sequences = ["AAAAAAAA","CCCCCCCC","DDDDDDDD","EEEEEEEE","FFFFFFFF","GGGGGGGG",
             "HHHHHHHH","IIIIIIII","KKKKKKKK","LLLLLLLL","MMMMMMMM","NNNNNNNN",
             "PPPPPPPP","QQQQQQQQ","RRRRRRRR","SSSSSSSS","TTTTTTTT","VVVVVVVV",
             "WWWWWWWW","YYYYYYYY"]


In [12]:
import numpy as np
from scipy.spatial.distance import pdist, squareform
import pandas as pd

atchley_df = pd.read_csv("../files/atchley.csv")


# Create a dictionary mapping amino acids to their Atchley factors
atchley_dict = atchley_df.set_index('amino.acid').to_dict(orient='index')

# Convert sequences into numerical representations
sequence_vectors = []
for seq in sequences:
    seq_matrix = np.array([list(atchley_dict[aa].values()) for aa in seq])  # Extract numerical values
    seq_vector = seq_matrix.flatten()  # Flatten to a single vector
    sequence_vectors.append(seq_vector)

# Compute pairwise Euclidean distances
distance_matrix = squareform(pdist(sequence_vectors, metric='sqeuclidean'))

# Convert to DataFrame for better readability
distance_df = pd.DataFrame(distance_matrix, index=sequences, columns=sequences)



In [13]:
distance_df


,AAAAAAAA,CCCCCCCC,DDDDDDDD,EEEEEEEE,FFFFFFFF,GGGGGGGG,HHHHHHHH,IIIIIIII,KKKKKKKK,LLLLLLLL,MMMMMMMM,NNNNNNNN,PPPPPPPP,QQQQQQQQ,RRRRRRRR,SSSSSSSS,TTTTTTTT,VVVVVVVV,WWWWWWWW,YYYYYYYY
AAAAAAAA,0.000000,83.395320,213.920664,90.415480,93.959984,145.477272,94.373712,92.025584,117.182128,12.460520,137.948488,121.708656,125.814648,127.606200,172.956992,245.671200,113.669648,23.934448,138.955280,227.887624
CCCCCCCC,83.395320,0.000000,184.485952,144.496464,77.108968,134.204000,35.937048,105.092952,137.971480,66.273392,128.507168,97.377432,71.315888,103.993232,209.806088,207.101992,138.955800,54.283112,34.826408,172.255424
DDDDDDDD,213.920664,184.485952,0.000000,283.554976,393.302152,468.744832,131.569336,450.737272,342.519000,146.588864,489.431216,338.227608,95.230768,21.265440,521.171832,42.180600,461.811000,175.146200,274.943864,555.533200
EEEEEEEE,90.415480,144.496464,283.554976,0.000000,66.560280,175.825120,121.063096,86.388616,65.910408,128.719120,80.674640,68.934504,191.178352,186.436704,128.299800,425.061176,87.123064,113.404952,96.354104,123.696800
FFFFFFFF,93.959984,77.108968,393.302152,66.560280,0.000000,84.293096,127.464144,7.208432,91.485360,129.673112,16.858024,51.928112,198.953336,264.265336,110.282208,474.345408,35.249936,92.992832,42.922080,51.824584
GGGGGGGG,145.477272,134.204000,468.744832,175.825120,84.293096,0.000000,198.005560,65.526680,98.876344,194.481504,127.068336,41.593080,172.763568,332.905632,61.579448,476.084728,25.957112,153.999448,146.386936,64.504816
HHHHHHHH,94.373712,35.937048,131.569336,121.063096,127.464144,198.005560,0.000000,170.022928,92.272656,83.138440,154.041400,107.830688,92.650072,50.226872,193.363952,194.736144,187.178192,102.969008,55.889712,217.970248
IIIIIIII,92.025584,105.092952,450.737272,86.388616,7.208432,65.526680,170.022928,0.000000,104.958592,137.684872,27.242152,61.457904,223.613576,313.193576,101.457632,514.912128,21.903488,98.207504,81.663952,56.608504
KKKKKKKK,117.182128,137.971480,342.519000,65.910408,91.485360,98.876344,92.272656,104.958592,0.000000,171.134952,85.680648,30.591696,192.600424,206.251560,26.839488,443.526112,68.771104,176.409104,104.096976,90.481880
LLLLLLLL,12.460520,66.273392,146.588864,128.719120,129.673112,194.481504,83.138440,137.684872,171.134952,0.000000,191.616784,163.817496,94.651536,85.761648,253.117432,162.233144,172.803608,13.191880,143.682728,291.346608


In [ ]:
import pandas as pd

# Create a DataFrame with sequences as index and columns
df = pd.DataFrame(distance_matrix, index=sequences, columns=sequences)

# Save to CSV
df.to_csv("out.csv")

In [3]:
from collections import defaultdict
import numpy as np

def truncate_sequence(sequence, left, right):
    mid = len(sequence) // 2
    return sequence[max(0, mid - left):min(len(sequence), mid + right)]

def compute_distance_matrix(sequences):
    """Placeholder function: Replace with your actual distance function."""
    n = len(sequences)
    dist_matrix = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            dist = sum(c1 != c2 for c1, c2 in zip(sequences[i], sequences[j])) / max(len(sequences[i]), len(sequences[j]))
            dist_matrix[i, j] = dist
            dist_matrix[j, i] = dist  # Symmetric
    return dist_matrix

def create_truncated_mapping(sequences, left, right):
    truncated_to_original = defaultdict(list)
    for seq in sequences:
        trunc_seq = truncate_sequence(seq, left, right)
        truncated_to_original[trunc_seq].append(seq)
    return truncated_to_original

def compute_truncated_distances(truncated_sequences, distance_func):
    """Compute the full distance matrix for truncated sequences."""
    return distance_func(truncated_sequences)

def expand_to_original_distances(truncated_to_original, truncated_dist_matrix, truncated_sequences):
    """Convert truncated sequence distances back to original sequences."""
    original_sequences = [seq for seqs in truncated_to_original.values() for seq in seqs]
    original_dist_matrix = np.zeros((len(original_sequences), len(original_sequences)))
    seq_to_idx = {seq: i for i, seq in enumerate(original_sequences)}
    
    for i, trunc1 in enumerate(truncated_sequences):
        for j, trunc2 in enumerate(truncated_sequences):
            dist = truncated_dist_matrix[i, j]
            for orig1 in truncated_to_original[trunc1]:
                for orig2 in truncated_to_original[trunc2]:
                    idx1, idx2 = seq_to_idx[orig1], seq_to_idx[orig2]
                    original_dist_matrix[idx1, idx2] = dist
                    original_dist_matrix[idx2, idx1] = dist  # Symmetric
    
    return original_dist_matrix, original_sequences

# Example usage:
original_sequences = ["CASSLDR", "CASSPGQ", "CASSLDQ", "CASRPGQ"]
left, right = 2, 2  # Adjust based on desired truncation

# Step 1: Create mapping
truncated_mapping = create_truncated_mapping(original_sequences, left, right)
truncated_sequences = list(truncated_mapping.keys())

# Step 2: Compute distance matrix for truncated sequences
truncated_dist_matrix = compute_truncated_distances(truncated_sequences, compute_distance_matrix)

# Step 3: Expand distances back to original sequences
original_dist_matrix, original_sequences = expand_to_original_distances(truncated_mapping, truncated_dist_matrix, truncated_sequences)

# Display results
print("Distance matrix for original sequences:")
print(original_dist_matrix)


Distance matrix for original sequences:
[[0.   0.   0.25 0.5 ]
 [0.   0.   0.25 0.5 ]
 [0.25 0.25 0.   0.25]
 [0.5  0.5  0.25 0.  ]]


In [2]:
from tqdm import tqdm
import heapq
import numpy as np

def prim_mst_with_progress(graph):
    n = len(graph)
    adj_list = {i: [] for i in range(n)}

    # Convert adjacency matrix to adjacency list
    for i in tqdm(range(n)):
        for j in range(i + 1, n):
            if graph[i, j] > 0:
                adj_list[i].append((graph[i, j], j))
                adj_list[j].append((graph[i, j], i))

    min_heap = [(0, 0)]
    visited = set()
    mst_edges = []
    total_cost = 0

    with tqdm(total=n, desc="Building MST") as pbar:
        while len(visited) < n:
            weight, node = heapq.heappop(min_heap)
            if node in visited:
                continue

            visited.add(node)
            total_cost += weight
            pbar.update(1)  # Update progress bar

            for edge_weight, neighbor in adj_list[node]:
                if neighbor not in visited:
                    heapq.heappush(min_heap, (edge_weight, neighbor))
                    mst_edges.append((node, neighbor, edge_weight))

    return mst_edges, total_cost

# Example: Generate a random adjacency matrix for 20,000 nodes
n = 20000
np.random.seed(42)
graph_matrix = np.random.rand(n, n)
np.fill_diagonal(graph_matrix, 0)

mst_edges, total_cost = prim_mst_with_progress(graph_matrix)
print("Final MST Cost:", total_cost)


Building MST: 100%|██████████| 20000/20000 [10:01<00:00, 33.23it/s]  


Final MST Cost: 1.2081034063959686


In [3]:
mst_edges

[(0, 1, 0.9507143064099162),
 (0, 2, 0.7319939418114051),
 (0, 3, 0.5986584841970366),
 (0, 4, 0.15601864044243652),
 (0, 5, 0.15599452033620265),
 (0, 6, 0.05808361216819946),
 (0, 7, 0.8661761457749352),
 (0, 8, 0.6011150117432088),
 (0, 9, 0.7080725777960455),
 (0, 10, 0.020584494295802447),
 (0, 11, 0.9699098521619943),
 (0, 12, 0.8324426408004217),
 (0, 13, 0.21233911067827616),
 (0, 14, 0.18182496720710062),
 (0, 15, 0.18340450985343382),
 (0, 16, 0.3042422429595377),
 (0, 17, 0.5247564316322378),
 (0, 18, 0.43194501864211576),
 (0, 19, 0.2912291401980419),
 (0, 20, 0.6118528947223795),
 (0, 21, 0.13949386065204183),
 (0, 22, 0.29214464853521815),
 (0, 23, 0.3663618432936917),
 (0, 24, 0.45606998421703593),
 (0, 25, 0.7851759613930136),
 (0, 26, 0.19967378215835974),
 (0, 27, 0.5142344384136116),
 (0, 28, 0.5924145688620425),
 (0, 29, 0.046450412719997725),
 (0, 30, 0.6075448519014384),
 (0, 31, 0.17052412368729153),
 (0, 32, 0.06505159298527952),
 (0, 33, 0.9488855372533332),
 (

In [23]:
sequence = "123456789123456"

mid = len(sequence) // 2
trunc_seq = sequence[max(0, mid - 4):min(len(sequence), mid + 4)]
print(trunc_seq)

mid = (len(sequence) + 1) // 2
trunc_seq = sequence[max(0, mid - 4):min(len(sequence), mid + 4)]
print(trunc_seq)

45678912
56789123
